# `webgpu` + `ngsolve_webgpu` - successor to webgui

GPU-native visualization for NGSolve, successor to webgui. The familiar
`Draw(cf, mesh, ...)`, or create your custom scene.

### webgpu
Python API for the JavaScript **WebGPU** standard (successor to WebGL, used by `webgui`).

- Basic foundations and general purpose rendering classes
- Camera, Colormap, Clipping, Fonts, ShapeRenderer
- No FEM-specific code (mesh, functions)

### ngsolve_webgpu

Renderers for FEM-specific objects:
- Mesh elements (surface, volume)
- Numbers (points, elements, facets, ...)
- CoefficientFunctions on surface elements, clipping plane, isosurface
- Isolines
- Vectors on surface/clipping plane

### Improvements vs webgui

WebGPU allows compute-only shaders and more flexible data storage on GPU. This removes some limitations of webgui:

- CoefficientFunctions with arbitrary order/number of components
- Compute shaders: run once, reuse during rendering (clipping plane, isosurface)
- Access multiple functions in one render draw (render a function at isosurface of another levelset function)

### Installation

```bash
pip install ngsolve_webgpu
````

Supported browsers: Chrome, Edge, Safari

Unsupported browsers: Firefox

On Linux: `about://flags`, enable "Unsafe WebGPU support" and "Vulkan"

### First triangle
The smallest possible scene: build a renderer, then `Draw` it.

In [ ]:
from webgpu.jupyter import Draw
from webgpu.triangles import TriangulationRenderer

tri = TriangulationRenderer([(0, 0, 0), (0, 1, 0), (1, 0, 0)], color=(1, 1, 0, 1))
Draw(tri)

In [ ]:
# Text rendering, compose multiple Renderer objects
from webgpu import Labels

# positions in screen coordinates x,y between -1 and 1
labels = Labels(["WebGPU", "is", "awesome"], positions=[(-0.7,0.8), (0.5, 0.3), (0, -0.5)], font_size=40)

Draw([tri, labels])

### Instancing & colormaps
Draw same shape (like arrows) at different positions, this is used by `SurfaceVectors` in `ngsolve_webgpu`.

In [ ]:
import numpy as np
from webgpu.shapes import generate_cylinder, generate_cone, ShapeRenderer
from webgpu.colormap import Colormap, Colorbar

# build an arrow from primitives (combine with +, reposition with .move())
shaft = generate_cylinder(n=12, radius=0.015, height=0.7, bottom_face=True)
tip   = generate_cone(n=12, radius=0.05, height=0.3, bottom_face=True).move((0, 0, 0.7))
arrow = shaft + tip

N = 48
theta = np.linspace(0, 2 * np.pi, N, endpoint=False)
positions  = np.c_[np.cos(theta), np.sin(theta), np.zeros(N)]      # on a circle
directions = np.c_[-np.sin(theta), np.cos(theta), np.zeros(N)]     # tangential
values     = np.repeat(theta, 2)                                   # 2 per object (start,end)

cmap = Colormap(colormap="viridis")        # plasma, matlab:jet, matplotlib:coolwarm, ...
cmap.set_min_max(float(values.min()), float(values.max()))
renderer = ShapeRenderer(arrow, positions=positions, directions=directions,
                         values=values, colormap=cmap)
Draw([renderer, Colorbar(cmap)])

### Clipping
`Clipping` cuts geometry on the GPU; the GUI auto-adds normal/offset sliders. Look inside a 3D mesh.

In [ ]:
from webgpu import Clipping, CoordinateAxes
clipping = Clipping(Clipping.Mode.PLANE, normal=(1,1,0))
renderer = ShapeRenderer(arrow, positions=positions, directions=directions,
                         values=values, colormap=cmap, clipping=clipping)
Draw([renderer, Colorbar(cmap), CoordinateAxes()])

### Compute shaders
Run a WGSL kernel on GPU buffers and read the result back - no canvas. Same compute path powers cuts, streamlines and isosurfaces later.

In [ ]:
from webgpu.utils import *   # numpy (np) imported above; webgpu runtime already initialized

device = get_device()
a = np.array([1, 2, 3], dtype=np.float32)
b = np.array([4, 5, 6], dtype=np.float32)
N = a.size
mem_size = a.size * a.itemsize

a_gpu = buffer_from_array(a)
b_gpu = buffer_from_array(b)
res_gpu = device.createBuffer(mem_size, BufferUsage.STORAGE | BufferUsage.COPY_SRC)
uniform_N = uniform_from_array(np.array([N], dtype=np.uint32))

bindings = [
    BufferBinding(101, a_gpu),
    BufferBinding(102, b_gpu),
    BufferBinding(103, res_gpu, read_only=False),
    UniformBinding(104, uniform_N),
]

shader_code = """
@group(0) @binding(101) var<storage> vec_a : array<f32>;
@group(0) @binding(102) var<storage> vec_b : array<f32>;
@group(0) @binding(103) var<storage, read_write> vec_res : array<f32>;
@group(0) @binding(104) var<uniform> N : u32;

@compute @workgroup_size(256, 1, 1)
fn main(@builtin(global_invocation_id) gid: vec3<u32>) {
    let tid = gid.x;
    if (tid < N) {
        vec_res[tid] = vec_a[tid] + vec_b[tid];
    }
}
"""

run_compute_shader(shader_code, bindings, n_workgroups=((N + 255) // 256, 1, 1))
print(read_buffer(res_gpu, np.float32))     # -> [5. 7. 9.]

## `ngsolve_webgpu` - the FEM specific stuff

The NGSolve-facing layer. Data lives on the GPU once and is shared by renderers; fields are
true high-order, evaluated on the GPU.


### The `Draw` you know
Mirrors the webgui `Draw` and dispatches on its first argument (Mesh / CF / OCC). High-order CF, colorbar and GUI come for free.

In [ ]:
from ngsolve_webgpu.jupyter import Draw    # high-level Draw(cf, mesh) — the webgui-style one
from ngsolve import unit_square, Mesh, sin, cos, x, y, CF
import ngsolve as ngs

mesh = Mesh(unit_square.GenerateMesh(maxh=0.15))
Draw(sin(10 * x) * cos(8 * y), mesh, order=3, deformation=CF((x, 0.3*x**2, 0)))

### `MeshData` + `FunctionData`
Data on the GPU once; `FunctionData` stores per-element Bernstein coefficients → true high-order rendering.

In [ ]:
from netgen.occ import *
from ngsolve_webgpu import FunctionData, MeshData, CFRenderer
from ngsolve_webgpu.jupyter import Draw          # back to the low-level renderer Draw

geo = OCCGeometry(Cylinder((0, 0, 0), X, r=0.3, h=0.5))
mesh = Mesh(geo.GenerateMesh(maxh=6))
mesh.Curve(5)                                  # high-order curved geometry

mesh_data = MeshData(mesh)
function_data = FunctionData(mesh_data, sin(20 * x), order=5)
cfr = CFRenderer(function_data)
cfr.colormap.set_min_max(-1, 1)
Draw([cfr, Colorbar(cfr.colormap)])

### Mesh visualization
- Entity numbers (vertices, segments, segment_indices, ....)
- 3D volume elements with a **shrink** slider and clipping; then a curved high-order surface mesh + wireframe.

In [ ]:
# Entity numbers (vertices) on the wireframe
from ngsolve_webgpu import EntityNumbers, MeshWireframe2d

mesh = Mesh(unit_square.GenerateMesh(maxh=0.3))
meshdata = MeshData(mesh)
Draw([MeshWireframe2d(meshdata), EntityNumbers(meshdata, entity="vertices", font_size=15)])

In [ ]:
from ngsolve_webgpu import MeshElements3d
mesh = Mesh(unit_cube.GenerateMesh(maxh=0.2))
meshdata = MeshData(mesh)
clipping = Clipping()
clipping.center = [0.5, 0.5, 0.5]
clipping.mode = clipping.Mode.PLANE

vol = MeshElements3d(meshdata, clipping=clipping)
vol.shrink = 0.8
numbers = EntityNumbers(meshdata, "segment_indices", clipping = clipping)
numbers.zero_based = False
Draw([vol, numbers], width=1000, height=700)

In [ ]:
# Curved surface mesh + wireframe (high order)
from ngsolve_webgpu import MeshElements2d, MeshWireframe2d

mesh = Mesh(OCCGeometry(Sphere((0, 0, 0), 1)).GenerateMesh(maxh=0.3))
mesh.Curve(2)
meshdata = MeshData(mesh)
Draw([MeshElements2d(meshdata), MeshWireframe2d(meshdata)])

### Clipping cross-sections
`ClippingCF` evaluates the field **on the cut plane** (compute shader).

In [ ]:
from ngsolve_webgpu import ClippingCF
from ngsolve import CF, x,y,z, sin,cos
mesh = Mesh(unit_cube.GenerateMesh(maxh=0.1))
cf = CF((sin(10 * z) * cos(15 * x), (x - 0.5)**2 + (y - 0.5)**2 + (z - 0.5)**2 - 2))

function_data = FunctionData(MeshData(mesh), cf, order=4)
clipping = Clipping()
clipping.mode = clipping.Mode.PLANE
clipping.center = [0.7, 0.5, 0.5]
clipping.normal = [1, -1, 1]

colormap = Colormap()
cfr  = CFRenderer(function_data, colormap=colormap, clipping=clipping)   # boundary surface
clip = ClippingCF(function_data, clipping=clipping, colormap=colormap)   # interior cut
Draw([cfr, clip, Colorbar(colormap)])

### Vector fields
- Arrows on a surface, arrows on a clipping plane are computed **on the GPU**
- Streamlines are computed on CPU (similar to webgui)

In [ ]:
# Arrows on the boundary surface
from ngsolve_webgpu import SurfaceVectors
mesh = Mesh(OCCGeometry(Sphere((0, 0, 0), 1)).GenerateMesh(maxh=0.3))
mesh.Curve(3)
mdata = MeshData(mesh)
fd = FunctionData(mdata, CF((x+0.1, 1j*y, -1j*z)), order=1)
Draw([SurfaceVectors(fd, grid_size=50)])

In [ ]:
# Arrows on a clipping-plane cross-section
from ngsolve_webgpu import ClippingVectors, MeshSegments

mesh = Mesh(unit_cube.GenerateMesh(maxh=0.2))
mdata = MeshData(mesh)
fd = FunctionData(mdata, CF((x, y, 1)), order=1)
clipping = Clipping()
clipping.center = [0.5, 0.5, 0.5]
segments = MeshSegments(mdata)
Draw([ClippingVectors(fd, clipping=clipping, grid_size=50), segments])

In [ ]:
from ngsolve_webgpu.cf import FieldLines

mesh = Mesh(unit_cube.GenerateMesh(maxh=0.3))
fieldlines = FieldLines(CF((-y, x, 0.1)), mesh,
                        num_lines=100, length=0.5, thickness=0.0015)
Draw([fieldlines, segments])

### Isosurfaces & isolines
Isosurface = zero level-set extracted by a compute shader, coloured by a second field. Isolines = contour lines in the fragment shader.

In [ ]:
# Isosurface (sphere level-set), coloured by x
from ngsolve_webgpu.isosurface import make_isosurface_renderers, IsoSurfaceRenderer

mesh = Mesh(OCCGeometry(Box((-1, -1, -1), (1, 1, 1))).GenerateMesh(maxh=0.2))
levelset = 0.5-(x**2 + y**2 + z**2)

mesh_data = MeshData(mesh)
func_data = FunctionData(mesh_data, x+y, order=1)
levelset_data = FunctionData(mesh_data, levelset, order=2)

segments = MeshSegments(mesh_data)
iso = IsoSurfaceRenderer(func_data, levelset_data)
Draw([iso, Colorbar(colormap), MeshSegments(mesh_data)], width=900)

In [ ]:
# Isosurface (sphere level-set), coloured by x
from ngsolve_webgpu.isosurface import make_isosurface_renderers

mesh = Mesh(OCCGeometry(Box((-1, -1, -1), (1, 1, 1))).GenerateMesh(maxh=0.2))
levelset = x**2 + y**2 + z**2

mesh_data = MeshData(mesh)
colormap = Colormap()
clipping = Clipping()
func_data = FunctionData(mesh_data, x, order=2)
levelset_data = FunctionData(mesh_data, levelset, order=2)

# One helper wires the isosurface + negative-surface + negative-clipping
# renderers to a single shared `value` uniform -> one "Level" slider in the GUI.
renderers, iso_settings = make_isosurface_renderers(func_data, levelset_data, clipping, colormap)
iso_settings.value = 0.5**2   # drive all three at once from Python
clipping.mode = clipping.Mode.PLANE
Draw(renderers + [Colorbar(colormap), MeshSegments(mesh_data)], width=900)

# Quick isolines via the high-level Draw

In [ ]:
from ngsolve_webgpu.jupyter import Draw

mesh = Mesh(unit_square.GenerateMesh(maxh=0.05))
Draw(sin(10 * x) * cos(10 * y), mesh, order=4, isolines=10)

In [ ]:
# IsolineRenderer with the coloured field underneath
from ngsolve_webgpu import IsolineRenderer
from webgpu.jupyter import Draw
from ngsolve import exp

mesh = Mesh(unit_square.GenerateMesh(maxh=0.05))
function_data = FunctionData(MeshData(mesh), exp(-(10 * ((x - 0.5)**2 + (y - 0.5)**2))), order=4)
renderer = IsolineRenderer(function_data, n_lines=12, thickness=2.0, show_field=True)
Draw([renderer, Colorbar(renderer.colormap)])

### Complex fields & phase
Real & imaginary parts uploaded interleaved. A **Complex Mode** dropdown + **Animate** checkbox appear.

In [ ]:
from ngsolve_webgpu.jupyter import Draw

mesh = Mesh(unit_square.GenerateMesh(maxh=0.1))
fes = ngs.H1(mesh, order=3, complex=True)
gf = ngs.GridFunction(fes)
gf.Set(sin(3 * x) + 1j * cos(3 * y))
scene = Draw(gf, mesh, order=3)

In [ ]:
# Switch the view (just a uniform write, no data re-upload) — needs the live kernel.
# Try "real" | "imag" | "abs" | "phase".  Or call animate_phase(scene, speed=0.2).
for ro in scene.render_objects:
    if isinstance(ro, CFRenderer):
        ro.set_complex_mode("abs")
        ro.animate_phase(scene, speed=0.2)   # smooth phase sweep
        break
scene.render()

### Element boundaries (DG / HDG)
Render a CF on element **facets** so inter-element jumps become visible - for discontinuous and facet-based spaces.

In [ ]:
# 2D: an L2 (discontinuous) function -> jumps across every edge
from ngsolve_webgpu import FacetFunctionData, FacetCFRenderer
from webgpu.jupyter import Draw

mesh = Mesh(unit_square.GenerateMesh(maxh=0.2))
mdata = MeshData(mesh)
gf = ngs.GridFunction(ngs.L2(mesh, order=0))
#gf = ngs.GridFunction(ngs.FacetFESpace(mesh, order=1))

gf.vec.FV().NumPy()[:] = [i / len(gf.vec) for i in range(len(gf.vec))]

facet_data = FacetFunctionData(mdata, gf, order=2)
colormap = Colormap()
Draw([FacetCFRenderer(facet_data, colormap=colormap, thickness=0.02), MeshWireframe2d(mdata), Colorbar(colormap)])

In [ ]:
# 3D: every tet face, slightly shrunk, with clipping
from ngsolve_webgpu import FacetCFRenderer3D

mesh = Mesh(unit_cube.GenerateMesh(maxh=0.3))
mdata = MeshData(mesh)

gf = ngs.GridFunction(ngs.L2(mesh, order=0))
gf.vec.FV().NumPy()[:] = [i / len(gf.vec) for i in range(len(gf.vec))]

colormap = Colormap()
clipping = Clipping()
clipping.mode = clipping.Mode.PLANE
clipping.center = [0.5, 0.5, 0.5]
clipping.normal = [1, 0, 0]

renderer = FacetCFRenderer3D(mdata, gf, order=1, colormap=colormap, clipping=clipping, shrink=0.9)
Draw([renderer, Colorbar(colormap)])

### Probe & pick
Entity numbers on the wireframe, and **click-to-pick** an element/region (picking needs the live kernel).

In [ ]:
# Click-to-pick: prints the element / region (needs the live kernel)
from ngsolve_webgpu import MeshPickResult
import webgpu.jupyter as wj

mesh = Mesh(unit_cube.GenerateMesh(maxh=0.3))
mesh_data = MeshData(mesh)
surface = MeshElements2d(mesh_data)
wireframe = MeshWireframe2d(mesh_data)

def on_pick(ev):
    print(MeshPickResult(ev, mesh, scene.options, kind="surface"))

surface.on_select(on_pick)
scene = wj.Draw([surface, wireframe])
scene.input_handler.on_click(lambda ev: scene.select(ev["canvasX"], ev["canvasY"]))

### Geometry from OCC
`Draw` takes an OCC shape directly: faces + edges + vertices with their real colours — no meshing required.

In [ ]:
from ngsolve_webgpu.jupyter import Draw

shape = Box((-1, -1, -1), (1, 1, 1)) - Cylinder((0, 0, -2), Z, r=0.5, h=4)
Draw(shape)

### Animation & live updates
`Animation` records per-frame GPU buffers; a **Frame** slider scrubs the timesteps and survives HTML export. Record frames *before* `Draw`.

In [ ]:
from ngsolve_webgpu.animate import Animation
from webgpu.jupyter import Draw

mesh = Mesh(unit_square.GenerateMesh(maxh=0.1))
gf = ngs.GridFunction(ngs.H1(mesh, order=3))
t = ngs.Parameter(0)
f = sin(10 * (x - 0.1 * t))
gf.Interpolate(f)

md_ = MeshData(mesh)
fd = FunctionData(md_, gf, order=3)
cfr = CFRenderer(fd)
cfr.colormap.set_min_max(-1, 1)

ani = Animation(cfr)
ani.add_time()                              # frame 0
for tval in range(1, 21):                   # record before Draw
    t.Set(tval * 5)
    gf.Interpolate(f)
    ani.add_time()

Draw([ani, Colorbar(cfr.colormap)])         # drag the "Animation/Frame" slider

### List all class names in ngsolve_webgpu

In [ ]:
import ngsolve_webgpu 
print('\n'.join([name for name in dir(ngsolve_webgpu) if name[0].isupper()]))

### Export to html
```bash
WEBGPU_EXPORTING=1 jupyter nbconvert --to html --execute your_notebook.ipynb
```

### Custom renderer in raw WGSL
A `Renderer` needs only `get_shader_code` + `get_bounding_box`. `#import camera` hands the vertex shader the interactive 3D camera.

In [ ]:
from webgpu.renderer import Renderer    # wj (webgpu.jupyter) imported earlier

shader_code = """
#import camera

struct FragmentInput {
    @builtin(position) p: vec4<f32>,
    @location(0) color: vec4<f32>,
};

@vertex
fn vertex_main(@builtin(vertex_index) vertex_index : u32) -> FragmentInput {
    var pos = array<vec3f, 3>(
        vec3f( 0.0,  0.5, 0.),
        vec3f(-0.5, -0.5, 0.),
        vec3f( 0.5, -0.5, 0.));
    var color = array<vec4f, 3>(
        vec4f(1., 0., 0., 1.),
        vec4f(0., 1., 0., 1.),
        vec4f(0., 0., 1., 1.));
    return FragmentInput(cameraMapPoint(pos[vertex_index]), color[vertex_index]);
}

@fragment
fn fragment_main(input: FragmentInput) -> @location(0) vec4f {
    return input.color;
}
"""

class TriangleCamera(Renderer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.n_vertices = 3
    def get_bounding_box(self):
        return ((-0.5, -0.5, 0), (0.5, 0.5, 0))
    def get_shader_code(self):
        return shader_code

wj.Draw(TriangleCamera())

## Links

### Documentation
- `ngsolve_webgpu`: <https://cerbsim.github.io/ngsolve_webgpu/>
- `webgpu`: <https://cerbsim.github.io/webgpu/>

### Source code
- source: <https://github.com/CERBSim/webgpu> · <https://github.com/CERBSim/ngsolve_webgpu>

### Interested? -> Hands-on session tomorrow!